In [23]:
# This is just for avoiding noise on the main code cell
import warnings
import sys
warnings.filterwarnings(action="ignore",category=UserWarning)
import astropy, numpy as np

print('python', sys.version)
print('numpy', np.__version__)
print('astropy', astropy.__version__)

python 3.13.7 (main, Mar  3 2026, 12:19:54) [GCC 15.2.0]
numpy 2.3.3
astropy 7.1.0


In [24]:
### CREATING A SERIES OF POINTS ALONG A SINGLE DAY

# handling time the right way
from datetime import datetime
from zoneinfo import ZoneInfo

current_tz = ZoneInfo('America/Sao_Paulo')
year, month, day = 2025, 11, 4
start_time = datetime(year, month, day, hour=7, tzinfo=current_tz)
end_time = datetime(year, month, day, hour=17, tzinfo=current_tz)

# Converting to numpy (on UTC) and creating a day line
# Note that numpy converts to UTC automatically it warns that this behavior will stop working
# The way it is done here is to futureproof the code.
utc = ZoneInfo('UTC')
np_start_time = np.datetime64(start_time.astimezone(utc), 'm')
np_time_delta = np.timedelta64(1,'m')                                # this sets the number of points (hour, minutes, second...)...
np_end_time  = np.datetime64(end_time.astimezone(utc), 'm') + np_time_delta
dayline = np.arange( np_start_time, np_end_time, np_time_delta)

# Setting the location to an specified point
from astropy.coordinates import EarthLocation, AltAz, get_sun
from astropy.units import deg
# lat, lon = -15.783507525862756*deg, -47.91526919090653*deg # praça do cruzeiro, Brasília, DF
lat, lon = -15.6007083*deg, -47.6594984*deg # FUP, Planaltina
location = EarthLocation(lat=lat, lon=lon)

# Converting the points to astropy Time
from astropy.time import Time
ap_dayline = Time(dayline)

# Build a reference frame on Earth Surface
from astropy.coordinates import AltAz, get_sun
frame = AltAz(location=location, pressure=0)

# Calculating the sun position for all specified points
sun = get_sun(ap_dayline).transform_to(frame)

sun_alt = sun.alt.value
sun_az = sun.az.value


In [25]:
ap_dayline[0], sun_alt[0], sun_az[0]

(<Time object: scale='utc' format='datetime64' value=2025-11-04T10:00:00.000000000>,
 np.float64(19.550627131117288),
 np.float64(101.27067023895549))

In [ ]:
# Calculating the shadow of the plumb
plumb_length = 1.0
shadow_length = plumb_length / np.tan(np.deg2rad(sun_alt))
x = - shadow_length * np.sin(np.deg2rad(sun_az))
y = shadow_length * np.cos(np.deg2rad(sun_az))

In [ ]:
np.min(np.abs(y))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(x,y)
plt.ylim(-2,2)
plt.show()

In [ ]:
# testing a grid

In [ ]:
### CREATING A DATETIMEARRAY

# handling time the right way
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
utc = ZoneInfo('UTC')

# Implement tests
def date_time_array(first_date: datetime,
                   last_date: datetime,
                   date_step: datetime,
                   time_range: timedelta = timedelta(hours=10,minutes=1),
                   time_step: timedelta = timedelta(minutes=1),
                   ):

    # Test edge cases

    first_date = np.datetime64( first_date.astimezone(utc))
    last_date =  np.datetime64( last_date.astimezone(utc))
    date_step =  np.timedelta64( date_step )

    days = np.arange(first_date, last_date, date_step)
    days = np.append(days, last_date)
    days_array = days.reshape( (len(days),1) ) # this turns a 1D array into a 2D, columnar, array

    trange = np.timedelta64(time_range)
    tstep = np.timedelta64(time_step)

    times = np.arange(0,trange, tstep)
    times_array = np.append(times, trange)

    return days_array + times_array

In [22]:
def last_solstice():
    return datetime(2025, 12, 21, hour = 7, tzinfo=current_tz)

def next_solstice():
    datetime(2026, 6, 21, hour = 7, tzinfo=current_tz)




In [32]:
date_time_array(first_date=datetime(year=2001,month=1,day=1,tzinfo=utc),
                last_date = datetime(year=2001,month=1,day=3,tzinfo=utc),
                date_step=timedelta(days=1),
                time_range=timedelta(hours=10),
                time_step=timedelta(hours=1),
)


array([['2001-01-01T00:00:00.000000', '2001-01-01T01:00:00.000000',
        '2001-01-01T02:00:00.000000', '2001-01-01T03:00:00.000000',
        '2001-01-01T04:00:00.000000', '2001-01-01T05:00:00.000000',
        '2001-01-01T06:00:00.000000', '2001-01-01T07:00:00.000000',
        '2001-01-01T08:00:00.000000', '2001-01-01T09:00:00.000000',
        '2001-01-01T10:00:00.000000'],
       ['2001-01-02T00:00:00.000000', '2001-01-02T01:00:00.000000',
        '2001-01-02T02:00:00.000000', '2001-01-02T03:00:00.000000',
        '2001-01-02T04:00:00.000000', '2001-01-02T05:00:00.000000',
        '2001-01-02T06:00:00.000000', '2001-01-02T07:00:00.000000',
        '2001-01-02T08:00:00.000000', '2001-01-02T09:00:00.000000',
        '2001-01-02T10:00:00.000000'],
       ['2001-01-03T00:00:00.000000', '2001-01-03T01:00:00.000000',
        '2001-01-03T02:00:00.000000', '2001-01-03T03:00:00.000000',
        '2001-01-03T04:00:00.000000', '2001-01-03T05:00:00.000000',
        '2001-01-03T06:00:00.000000', 